# 01 — Explorer un corpus annoté au niveau du caractère

**Projet** : Extraction d'entités nommées par spaCy — le tagger appris, corroboré par des règles

## Objectifs pédagogiques

1. Lire les **deux** tables d'un projet d'extraction : les messages, et les mentions annotées dans ces messages.
1. Mesurer ce qu'une **règle de forme** retrouverait déjà, type par type : le raccourci que la F1 micro cache.
1. Repérer les **surfaces réservées** à l'évaluation et les distracteurs déclarés.
1. Comparer le corpus des notebooks à la référence publiée par le projet.

## 1. Mise en place

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.data.schemas import (  # noqa: E402
    ENTITY_LABELS,
    NAME_LABELS,
)
from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un corpus réduit (320 messages) et un budget d'époques
# réduit : l'exécution reste rapide, et les valeurs absolues d'un notebook ne sont pas celles de la
# référence publiée par le README.
NB_DOCUMENTS = 320
NB_EPOCHS = 3

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                f"train.epochs={NB_EPOCHS}",
                "seed=42",
                "log_level=WARNING",
                "train.callbacks.progress_bar=false",
                "train.callbacks.logging_every=100",
            ],
        )
    )

MODEL_NODE = dict(node(CONFIG, "model"))
TEXT_COLUMN = str(MODEL_NODE.get("text_column", "text"))
TARGET_COLUMN = str(MODEL_NODE.get("target", "label"))
ID_COLUMN = str(CONFIG.data.id_column or "msg_id")
SPLIT_COLUMN = str(CONFIG.data.group_column or "split")

# Les notebooks lisent **et écrivent** dans leur propre bac à sable : le corpus réduit et les
# artefacts des notebooks ne touchent ni `data/raw` ni `artifacts/`, donc ceux de `make train`
# restent ceux de la référence.
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=PROJECT_ROOT / "outputs" / "notebooks" / "data",
    artifacts_dir=PROJECT_ROOT / "outputs" / "notebooks" / "artifacts",
).ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil contractuel : 0.7)")
print(f"Algorithme servi  : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Types d'entités   : {len(ENTITY_LABELS)} — {', '.join(ENTITY_LABELS)}")
print(f"Bac à sable       : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

In [ ]:
from src.data.generators import SyntheticEntityCorpusGenerator
from src.data.loaders import EntityCorpusLoader

LOADER = EntityCorpusLoader(
    NB_PATHS,
    dataset_name=str(CONFIG.data.dataset_name),
    formats=tuple(CONFIG.data.formats),
    validation_enabled=bool(CONFIG.data.validation.raw),
    lazy_validation=bool(CONFIG.data.validation.lazy),
)

if LOADER.documents_path.exists():
    DOCUMENTS, SPANS = LOADER.load_corpus()
    METADATA = LOADER.load_metadata()
    print(f"Corpus lu depuis le bac à sable : {len(DOCUMENTS)} messages, graine {METADATA['seed']}")
else:
    # Un clone frais n'a pas encore de données : le notebook reste exécutable. Le corpus réduit est
    # persisté **dans le bac à sable** pour que les notebooks suivants et les pipelines travaillent
    # sur exactement les mêmes fichiers.
    bundle = SyntheticEntityCorpusGenerator.from_config(CONFIG.data).generate()
    DOCUMENTS, SPANS, METADATA = bundle.documents, bundle.queries, dict(bundle.metadata)
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_annotations(SPANS)
    LOADER.save_metadata(METADATA)
    print(f"Bac à sable vide : corpus de {len(DOCUMENTS)} messages généré et persisté")

DOCUMENTS = DOCUMENTS.sort_values(ID_COLUMN).reset_index(drop=True)
SPANS = SPANS.sort_values([ID_COLUMN, "start", "end"]).reset_index(drop=True)

TRAIN_DOCUMENTS = LOADER.split("train", frame=DOCUMENTS)
VAL_DOCUMENTS = LOADER.split("val", frame=DOCUMENTS)
CALIBRATION_DOCUMENTS = LOADER.split("calibration", frame=DOCUMENTS)
TEST_DOCUMENTS = LOADER.split("test", frame=DOCUMENTS)
TRAIN_SPANS = LOADER.split_annotations("train", documents=DOCUMENTS, spans=SPANS)
VAL_SPANS = LOADER.split_annotations("val", documents=DOCUMENTS, spans=SPANS)
TEST_SPANS = LOADER.split_annotations("test", documents=DOCUMENTS, spans=SPANS)

print(
    "messages par split :",
    {
        name: len(frame)
        for name, frame in (
            ("train", TRAIN_DOCUMENTS),
            ("val", VAL_DOCUMENTS),
            ("calibration", CALIBRATION_DOCUMENTS),
            ("test", TEST_DOCUMENTS),
        )
    },
)
print(
    "mentions par split :",
    {
        name: len(frame)
        for name, frame in (("train", TRAIN_SPANS), ("val", VAL_SPANS), ("test", TEST_SPANS))
    },
)

## 2. Le corpus en chiffres

Un corpus de reconnaissance d'entités se décrit par ses **mentions**, pas par ses lignes : le nombre de messages ne dit rien de la difficulté, la répartition des types et la longueur des mentions, oui.

In [ ]:
per_message = SPANS.groupby("msg_id").size()
print(f"messages          : {len(DOCUMENTS)}")
print(f"mentions annotées : {len(SPANS)}")
print(
    f"par message       : {per_message.mean():.2f} en moyenne "
    f"(min {per_message.min()}, max {per_message.max()})"
)
print()
print("métadonnées de génération — tailles :", METADATA["entity_counts"])
print("styles :", METADATA["style_shares"], "| canaux :", METADATA["canal_shares"])

In [ ]:
from src.data.schemas import describe_labels, holdout_share, label_counts

comptes = label_counts(SPANS)
reserves = holdout_share(SPANS)
carte_types = pd.DataFrame(
    {
        "type": list(ENTITY_LABELS),
        "mentions": [comptes[label] for label in ENTITY_LABELS],
        "part du corpus": [round(comptes[label] / len(SPANS), 4) for label in ENTITY_LABELS],
        "part réservée": [round(reserves[label], 4) for label in ENTITY_LABELS],
        "famille": ["nom" if label in NAME_LABELS else "motif" for label in ENTITY_LABELS],
        "définition": [describe_labels()[label] for label in ENTITY_LABELS],
    }
)
display(carte_types)
print(f"part réservée au total : {SPANS['holdout'].mean():.2%} des mentions")

**Ce qu'il faut retenir**

- Les types sont déséquilibrés : la F1 **micro** les noie, la F1 **macro** les compte à égalité — les deux sont publiées pour cette raison.
- La colonne « part réservée » est la difficulté du type : une surface de produit jamais vue à l'entraînement ne peut pas être retrouvée par une liste apprise.
- Trois types s'écrivent selon un motif (commande, montant, date), deux reposent sur un nom (produit, transporteur). Tout le reste du projet découle de cette distinction.

## 3. Ce que la forme seule explique déjà

Avant de juger un modèle, il faut savoir ce qu'une règle d'une ligne obtiendrait. La fonction `shape_shortcut_scores` apprend, **sur le train**, la signature de forme dominante de chaque type, puis classe avec sur la validation — une forme jamais vue n'est reconnue par personne.

In [ ]:
from src.features.build_features import shape_separability, shape_shortcut_scores

raccourci = shape_shortcut_scores(DOCUMENTS, SPANS, split="val").set_index("label")
display(
    raccourci[
        [
            "n_spans",
            "n_signatures",
            "n_unseen",
            "signature",
            "share_by_shape",
            "precision",
            "recall",
            "f1",
        ]
    ]
    .rename(
        columns={
            "n_spans": "mentions (val)",
            "n_signatures": "formes distinctes",
            "n_unseen": "formes inédites",
            "signature": "forme dominante (train)",
            "share_by_shape": "part classée",
        }
    )
    .round(4)
)

In [ ]:
separabilite = shape_separability(DOCUMENTS, SPANS).sort_values("n_spans", ascending=False)
ambiguës = separabilite[separabilite["n_labels"] > 1]
print(f"signatures de forme observées : {len(separabilite)}")
print(f"signatures partagées par plusieurs types : {len(ambiguës)}")
display(ambiguës.head(10).round(4))

**Ce qu'il faut retenir**

- La règle de forme atteint 0,93 à 1,00 de F1 sur **chaque** type de la validation, dès que la forme a été vue : sur ce corpus, une signature ne porte qu'un seul type (`n_labels` = 1 partout) — la forme *nomme* une mention déjà bornée, elle ne la trouve pas.
- `n_unseen` compte les mentions de validation dont la signature est absente du train (ici le produit) : c'est exactement ce que la surface réservée provoque, et ce qu'aucune liste de surfaces vue à l'entraînement ne peut rattraper.
- La difficulté du projet n'est donc pas de choisir entre deux types sur la même forme, mais de **trouver** les bornes des mentions et de reconnaître des formes inédites.

## 4. Distracteurs déclarés et plancher trivial

Un corpus pédagogique doit **déclarer** ce qui y ressemble à une entité sans en être une. Villes, numéros de facture et familles de produits apparaissent dans les textes et ne sont jamais annotés : un modèle qui les annote perd de la précision.

In [ ]:
villes = list(METADATA["distractors"]["cities"])
presentes = [
    ville for ville in villes if DOCUMENTS[TEXT_COLUMN].str.contains(ville, regex=False).any()
]
annotees = sorted(set(SPANS["surface"]) & set(villes))
print("villes présentes dans les textes :", presentes)
print("villes annotées comme entités   :", annotees or "aucune (déclarées distrayantes)")
print("faux positifs que cela promet   :", len(presentes), "pièges à ne pas récompenser")
print()
print("plancher trivial :", METADATA["trivial_floor"])

In [ ]:
from src.visualization.plots import plot_entity_counts, plot_lengths

DISTRIBUTION = LOADER.entity_distribution()
figure_types = plot_entity_counts(DISTRIBUTION, NB_PATHS.figures_dir / "eda_entity_counts.png")
figure_longueurs = plot_lengths(DOCUMENTS, NB_PATHS.figures_dir / "eda_lengths.png")
display(Image(filename=str(figure_types)))
display(Image(filename=str(figure_longueurs)))

## 5. Ce que la référence du projet a mesuré

Le README publie les chiffres du **run de référence** (corpus complet, budget d'époques du manifeste). Les valeurs de ce notebook sont celles du corpus réduit : elles servent à comprendre, pas à citer.

In [ ]:
REFERENCE_RESULTS = [
    (
        "F1 au niveau entité, micro, sur le split de test (seuil 0,7 ; plancher trivial 0,0)",
        "0,9340",
    ),
    ("Précision / rappel au niveau entité (bornes exactes exigées)", "0,9521 / 0,9167"),
    (
        "F1 macro (les cinq types comptent pareil) / F1 partielle (bornes tolérées)",
        "0,9239 / 0,9735",
    ),
    ("Référence « couche de règles » sur le même test (surfaces du train uniquement)", "0,8710"),
    (
        "Rappel sur les surfaces réservées aux splits d'évaluation / déjà vues au train",
        "0,6056 / 1,0000",
    ),
    ("Confiance moyenne publiée (écart sur les mentions correctes)", "0,8423 (0,1396)"),
    ("Erreurs classées au test (inventée / bornes / manquée)", "5 / 20 / 0"),
    (
        "Type le plus difficile (F1 au niveau entité) / le plus facile",
        "transporteur 0,713 / commande, date, montant 1,000",
    ),
    (
        "Surfaces réservées au test : produits et transporteurs jamais vus à l'entraînement",
        "12 / 4",
    ),
]
if REFERENCE_RESULTS:
    display(pd.DataFrame(REFERENCE_RESULTS, columns=["mesure", "valeur"]))
else:
    print("Aucune mesure publiée dans le manifeste : voir la section du README du projet.")
print()
print(
    f"corpus du notebook : {len(DOCUMENTS)} messages, {len(SPANS)} mentions, "
    f"budget d'entraînement {NB_EPOCHS} époque(s) par défaut"
)
print(f"métrique principale visée : {CONFIG.metrics.primary} >= {CONFIG.metrics.min_primary}")

**Ce qu'il faut retenir**

- Deux tables, un contrat : les annotations ne vivent pas dans le texte, elles pointent dessus par des offsets — c'est ce que le notebook 02 vérifie.
- Un corpus synthétique n'est pas un corpus facile : les surfaces réservées et les distracteurs sont là pour que les scores veuillent dire quelque chose.
- Le plancher trivial (0,0) est publié pour que tout score se lise par rapport à lui, jamais dans l'absolu.

## Synthèse

| Question | Réponse lue ici | Suite |
| --- | --- | --- |
| Combien de mentions, et de quel type ? | `label_counts`, `entity_distribution` | notebook 06 (F1 par type) |
| Quelle part de chaque type est un **motif** ? | `shape_shortcut_scores` | notebook 03 (traits de forme) |
| Quelle part est réservée à l'évaluation ? | `holdout_share` | notebook 06 (écart surface réservée / vue) |
| Les tables sont-elles conformes ? | — | `02_validation.ipynb` |

**Suite** : `02_validation.ipynb` fait échouer les contrats pour montrer ce qu'ils protègent.